In [ ]:
import gc
import os
import sys
from dataclasses import dataclass
# from google.colab import userdata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pprint import pprint
from typing import Callable
from datetime import datetime

import random
# from tqdm import tqdm
import time
from tqdm.autonotebook import tqdm as notebook_tqdm
import torch
from torch.utils.data import Dataset, DataLoader
# torch.set_float32_matmul_precision('high')

from huggingface_hub import login
import transformers
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, AutoConfig, logging, AutoModelForMaskedLM
from langchain_text_splitters import RecursiveCharacterTextSplitter

import warnings
warnings.filterwarnings('ignore')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
sys.version

In [ ]:
class Dataset10x(Dataset):
    def __init__(self, report_df: pd.DataFrame, items_path: str = 'items') -> None:
        super().__init__()

        self.target_columns = ['FILING_DATE', 'CIK', 'ACC_NUM']
        for col in self.target_columns:
            assert col in report_df.columns, col

        self.name_columns = [self.target_columns[0], '10-K_edgar_data'] + self.target_columns[1:]
        self.item_names = ['item1', 'item1a', 'item7']
        self.items_path = items_path
        self.report_df = report_df

    def __len__(self) -> int:
        return len(self.report_df)

    def __getitem__(self, idx: int) -> dict[str, str]:
        report_dict = {}
        notnone = False
        report = self.report_df.iloc[idx].to_dict()
        report_name = '_'.join([str(report[x]) if x in report.keys() else x for x in self.name_columns])

        for item_name in self.item_names:
            item_pathname = os.path.join(self.items_path, f'{item_name}_files', f'{report_name}_{item_name}.txt')

            if not os.path.exists(item_pathname):
                item_pathname = os.path.join(self.items_path, f'{report_name}_{item_name}.txt')

            if os.path.exists(item_pathname):
                with open(item_pathname, 'r', encoding='utf-8') as file:
                    item_text = file.read()
                    report_dict[item_name] = item_text
                    notnone = True
        return (idx, report_dict) if notnone else (idx, None)

def collate_fn_item7(batch):
    indices = []
    item7_texts = []
    
    for idx, data in batch:
        if data is not None and "item7" in data:
            item7 = data["item7"]
#             if len(item7) > 200:
            indices.append(idx)
            item7_texts.append(item7)
    return indices, item7_texts

In [ ]:
def fill_prompt_batch(texts, ending, prompt_func, tokenizer, model,
                      top_k=1, top_p=None, text_length=1000, verbose=False, device='cuda'):
    """
    - Generates tokens for a batch of texts
    - Uses model to obtain the logits (prob to fill mask)
    - Returns the top_k tokens and associated probs
    """
    clean_mem()

    prompt_texts = [prompt_func(text[:text_length], ending) for text in texts]
    
    inputs = tokenizer(prompt_texts, return_tensors="pt", truncation=True,
                       padding='max_length').to(device)
    
    if verbose:
        print(inputs['input_ids'].shape)

    with torch.no_grad():
        logits = model(**inputs).logits.cpu()

#     probss = torch.nn.functional.softmax(logits[:,-1,:], dim=-1)
    pad_ix = (inputs["input_ids"]==tokenizer.mask_token_id).cpu()    # batch_size, vocab_size
    answer_logits = logits[pad_ix]
    answer_probs = torch.nn.functional.softmax(answer_logits, dim=-1)

    del inputs
    gc.collect()
    
    if top_p is not None and top_p>0 and top_p<=1.0:
        sorted_probs, sorted_indices = torch.sort(answer_probs, descending=True, dim=-1)    # batch_size by vocab_size
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        # Remove tokens with cumulative probability above the threshold
        sorted_indices_to_keep = cumulative_probs <= top_p                           # top_p
        # Shift the indices to the right to keep also the first token above the threshold
        sorted_indices_to_keep[..., 1:] = sorted_indices_to_keep[..., :-1].clone()
        sorted_indices_to_keep[..., 0] = True
        indices_to_keep = [sorted_indices[i][sorted_indices_to_keep[i]].tolist() for i in range(len(sorted_indices))]
        probs_to_keep = [sorted_probs[i][sorted_indices_to_keep[i]].tolist()  for i in range(len(sorted_indices))]
        answers = [dict(zip(l.strip().split(),probs_to_keep[i]))
                  for i,l in enumerate(tokenizer.batch_decode(indices_to_keep))]
        return answers
    
    else:
        top_k_tokens = torch.topk(answer_probs, top_k, dim=1).indices.tolist()
        top_k_probs = torch.topk(answer_probs, top_k, dim=1).values.tolist()
        top_k_decoded = tokenizer.batch_decode(top_k_tokens)
        return [dict(zip(d.strip().split(" "), top_k_probs[i])) for i,d in enumerate(top_k_decoded)]

In [ ]:
def apply_strategy(texts, ending, strategy, tokenizer, model, text_length=5000, verbose=False, device='cuda'):
    '''
    Applies prompt strategy

    - text: to be appended the prompt
    - strategy: contains the prompt and the verbalizer
    '''

    output = fill_prompt_batch(texts, ending, strategy.prompt, tokenizer=tokenizer,
                               model=model, top_p=strategy.top_p, text_length=text_length, device=device)
    clean_mem()

    scores = []
    for item in output:
        score = dict()
        for cat, vals in strategy.verbalizer.items():
            # Strip spaces + turn lowercase, then match with verbalizer
            score[cat] = sum([v for k,v in item.items() if k.strip().lower() in vals])
            try:
                score['polarity'] = score['positive'] - score['negative']
            except Exception:
                pass
        scores.append(score)

    if verbose:
        print(scores)
    return scores

In [ ]:
def get_model_output(prompt, model, device, k=20) -> dict[str, float]:

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True,
                     padding='max_length').to(device)

    print(f"input has {inputs['input_ids'].shape} tokens")

    with torch.no_grad():
        logits = model(**inputs).logits.cpu()

#     logits = logits[0,-1,:]
#     probs = torch.nn.functional.softmax(logits, dim=-1)

    pad_ix = (inputs["input_ids"]==tokenizer.mask_token_id).cpu()    # batch_size, vocab_size
    answer_logits = logits[pad_ix]
    answer_probs = torch.nn.functional.softmax(answer_logits, dim=-1)
    
    top_k_tokens = torch.topk(answer_probs, k, dim=1)
    top_ix = top_k_tokens.indices.tolist()[0]
    top_prob = top_k_tokens.values.tolist()[0]

    # return dict(zip(top_ix, top_prob))
    return dict(zip([tokenizer.decode([x]) for x in top_ix], top_prob))

In [ ]:
def clean_mem():
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    time.sleep(0.1)

In [ ]:
def gather_stats(strategy, results, tokenizer, model, data, ending, verbose=False, text_length=5000, 
                save_path="results", save_interval=5000, resume=True, max_retries=3, device='cuda'):
    
    if resume and results:
        max_processed_idx = max([df.index.max() for df in results if not df.empty])
        print(f"Resuming from index: {max_processed_idx}")
    else:
        max_processed_idx = -1
    
    processed_count = 0
    batch_count = 0
    
    for doc_indices, batch_chunks, chunks_per_doc in notebook_tqdm(data):
        if len(doc_indices) > 0:
            if resume and max(doc_indices) <= max_processed_idx:
                continue
            else:
                success = False
                retry_count = 0

                while not success and retry_count < max_retries:
                    try:
                        # Применяем стратегию ко всем чанкам
                        chunk_scores = apply_strategy(batch_chunks, ending, strategy=strategy, 
                                                     tokenizer=tokenizer, model=model, 
                                                     text_length=text_length, device=device)
                        clean_mem()
                        
                        # Агрегируем результаты по документам
                        doc_scores = []
                        start_idx = 0
                        for doc_idx, num_chunks in zip(np.unique(doc_indices), chunks_per_doc):
                            if num_chunks == 0:
                                continue
                            
                            # Получаем скоры для всех чанков этого документа
                            doc_chunk_scores = chunk_scores[start_idx:start_idx + num_chunks]
                            start_idx += num_chunks
                            
                            # Агрегируем скоры (например, среднее по всем чанкам)
                            aggregated_score = {}
                            for key in doc_chunk_scores[0].keys():
                                values = [score[key] for score in doc_chunk_scores if key in score]
                                aggregated_score[key] = sum(values) / len(values) if values else 0
                            
                            doc_scores.append((doc_idx, aggregated_score))
                        
                        # Сохраняем результаты
                        indices = [idx for idx, _ in doc_scores]
                        scores = [score for _, score in doc_scores]
                        
                        df = pd.DataFrame(scores, index=indices)
                        results.append(df)
                        processed_count += len(indices)
                        batch_count += 1

                        success = True

                    except torch.cuda.OutOfMemoryError as oom_error:
                        retry_count += 1
                        print(f"CUDA OOM error processing batch {doc_indices} (attempt {retry_count}): {oom_error}")
                        
                        clean_mem()
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        
                        if retry_count >= max_retries:
                            print(f"Failed to process batch after {max_retries} attempts")
                            break
                    
                    except Exception as e:
#                         print(f"Error processing batch with indices {doc_indices}: {e}")
                        break
                
                # Сохранение промежуточных результатов
                if processed_count >= save_interval:
                    try:
                        stats_df = pd.concat(results, axis=0)
                        stats_df.reset_index(inplace=True)
                        
                        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                        csv_file = os.path.join(save_path, f"results_{timestamp}_{processed_count}_reports.csv")
                        stats_df.to_csv(csv_file, index=False)
                        
                        processed_count = 0
                        
                    except Exception as e:
                        print(f"Error saving intermediate results: {e}")
        
        else:
            continue
    
    # Финальное сохранение
    if results:
        try:
            stats_df = pd.concat(results, axis=0)
            stats_df.reset_index(inplace=True)
            
            final_csv_file = os.path.join(save_path, "final_results.csv")
            stats_df.to_csv(final_csv_file, index=False)
            
            if verbose:
                print(stats_df.info())
            
            return stats_df
        
        except Exception as e:
            print(f"Error saving final results: {e}")
            return pd.DataFrame()
    else:
        print("No results to process")
        return pd.DataFrame()

In [ ]:
def calculate_mean_polarities(tech_res, blue_chip_res):
    result_dict = {}
    tech_sent_before_crisis = tech_res[tech_res['FILING_DATE'].astype(str).str[:4].between('1997', '1999')].reset_index(drop=True)
    tech_sent_after_crisis = tech_res[tech_res['FILING_DATE'].astype(str).str[:4].between('2000', '2003')].reset_index(drop=True)

    blue_chip_before_crisis = blue_chip_res[blue_chip_res['FILING_DATE'].astype(str).str[:4].between('1997', '1999')].reset_index(drop=True)
    blue_chip_after_crisis = blue_chip_res[blue_chip_res['FILING_DATE'].astype(str).str[:4].between('2000', '2003')].reset_index(drop=True)
    
    tech_line = [tech_sent_before_crisis['polarity'].mean(), tech_sent_after_crisis['polarity'].mean()]
    result_dict['tech'] = {}
    result_dict['tech']['before'] = tech_line[0]
    result_dict['tech']['after'] = tech_line[1]

    blue_chip_line = [blue_chip_before_crisis['polarity'].mean(), blue_chip_after_crisis['polarity'].mean()]
    delta = blue_chip_line[0] - blue_chip_line[1]
    result_dict['blue_chip'] = {}
    result_dict['blue_chip']['before'] = blue_chip_line[0]
    result_dict['blue_chip']['after'] = blue_chip_line[1]
    
    res_df = pd.DataFrame.from_dict(result_dict).T
    res_df['delta'] = res_df['before'] - res_df['after']
    res_df = round(res_df, 4)
    display(res_df)
    
    plt.title('Polarity general ModernBERT', weight='bold')
    plt.plot(tech_line, marker='o', label='tech')
    plt.plot(blue_chip_line, marker='o', label='non-tech')

    # for i in range(len(tech_line)):
    #     plt.text(i, tech_line[i], f'  {tech_line[i]:.3f}', va='bottom', ha='left', fontsize=9)
    #     plt.text(i, blue_chip_line[i], f'  {blue_chip_line[i]:.3f}', va='bottom', ha='left', fontsize=9)

    plt.ylabel('mean sentiment')
    plt.xticks(ticks=[0, 1], labels=['before crisis (1997 -- 1999)', 'after crisis (2001 -- 2003)'])
    plt.legend()
    plt.grid()

    return tech_line, blue_chip_line, result_dict

def calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, sentiment_verb, text_length=10_000):
    sentiment_strategy = Prompt_Strategy('sentiment', sentiment_verb, get_prompt)
    
    results = []

    blue_chip_sent = gather_stats(
        sentiment_strategy, results=results, tokenizer=tokenizer, model=model,
        data=blue_chip_dataloader, ending=ending, verbose=False, text_length=text_length, 
        save_path="/home/jovyan/datavol-1/project_10x/speculative_sent/blue_chip_sentiment/",
        save_interval=2000, resume=True, max_retries=3, device=device
    )

    blue_chip_res = blue_chip_df.join(blue_chip_sent)
    clean_mem()
    
    results = []

    tech_sent = gather_stats(
        sentiment_strategy, results=results, tokenizer=tokenizer, model=model,
        data=tech_dataloader, ending=ending, verbose=False, text_length=text_length, 
        save_path="/home/jovyan/datavol-1/project_10x/speculative_sent/tech_sentiment/",
        save_interval=2000, resume=True, max_retries=3, device=device
    )

    tech_res = tech_df.join(tech_sent)
    clean_mem()
    
    return tech_res, blue_chip_res

# Data

In [ ]:
df = pd.read_sas('/home/jovyan/datavol-1/project_10x/finaldata_10k.sas7bdat')

df['ACC_NUM'] = df['filename'].astype(str).str.split('_').str[1]
df['CIK'] = df['filename'].astype(str).str.split('_').str[0]
df['FILING_DATE'] = df['date_filed'].astype(str).str.replace('-', '')
reports = df
reports
# reports = df[df['FILING_DATE'].astype(str).str[:4] < '2013'].reset_index(drop=True)
# reports = reports[['CIK', 'FILING_DATE', 'ACC_NUM']]
# reports

# df = pd.read_csv('/home/jovyan/datavol-2/missing_item7.tsv.gz', sep='\t')
# df['ACC_NUM'] = df['name'].astype(str).str.split('_').str[-1]
# df['CIK'] = df['name'].astype(str).str.split('_').str[-2]
# df['FILING_DATE'] = df['name'].astype(str).str.split('_').str[0]
# reports = df[['CIK', 'FILING_DATE', 'ACC_NUM']]
# reports

In [ ]:
summary = pd.read_csv('/home/jovyan/datavol-1/project_10x/Loughran-McDonald_10X_Summaries_1993-2023.csv')

tech_names = ['CISCO', 'AMAZON ', 'ETOYS', 'INTEL ', 'NETSCAPE', 'PETS ', 'NETSCAPE', 'YAHOO', 'FLOOZ', 'GEEKNET',
            'INTERACTIVE INTELLIGENCE', 'NETWORK SOLUTIONS', 'PALM', 'PIXELON', 'QUALCOMM', 'STEEL CONNECT', 'TRANSMETA',
            'UUNET', 'VERIO', 'TIBCO', 'UBID', 'WEBVAN']

blue_chip_names = ['COCA COLA', 'AT&T', 'PHILIP MORRIS', 'DUPONT', 'MERCK', 'GENERAL ELECTRIC']

selected_columns = ['CIK', "FILING_DATE", 'ACC_NUM', 'CoName']

tech_df = summary[summary['CoName'].str.startswith(tuple(tech_names))][selected_columns].reset_index(drop=True)
tech_df = tech_df[tech_df['FILING_DATE'].astype(str).str[:4].between('1997', '2003')].reset_index(drop=True)

blue_chip_df = summary[summary['CoName'].str.startswith(tuple(blue_chip_names))][selected_columns].reset_index(drop=True)
blue_chip_df = blue_chip_df[blue_chip_df['FILING_DATE'].astype(str).str[:4].between('1997', '2003')].reset_index(drop=True)

In [ ]:
items_path = '/home/jovyan/datavol-1/project_10x/items/item7_files'
# items_path = '/home/jovyan/datavol-2/missing_item7'

tech_dataset = Dataset10x(tech_df, items_path)

not_none = []
for idx in range(393):
    item_dict = tech_dataset[idx]
    if item_dict[1] is not None:
        not_none.append(idx)
        
# len(not_none)
tech_df = tech_df.iloc[not_none].reset_index(drop=True)

In [ ]:
blue_chip_dataset = Dataset10x(blue_chip_df, items_path)

not_none = []
for idx in range(402):
    item_dict = blue_chip_dataset[idx]
    if item_dict[1] is not None:
        not_none.append(idx)
        
blue_chip_df = blue_chip_df.iloc[not_none].reset_index(drop=True)

In [ ]:
blue_chip_df.shape, tech_df.shape

# Model - ModernBERT

In [ ]:
model_id = "/home/jovyan/models/ModernBERT-large"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_id,
#                                           max_length=length,
                                          padding=True,
                                          truncation=True,
                                          cache_dir="hf_cache")
if not tokenizer.pad_token:
    print("Adding pad token")
    tokenizer.pad_token = tokenizer.eos_token
    
tokenizer.padding_side = "left"

model = AutoModelForMaskedLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, cache_dir="hf_cache"
).to(device)

In [ ]:
config = AutoConfig.from_pretrained(model_id)
length = config.max_position_embeddings
length

In [ ]:
item_name = 'item7'

q99 = 275568
q95 = 169981

In [ ]:
def get_prompt(item_text: str, ending: str) -> str:
    return item_text + "\n\n" + ending 

## Dataloader

In [ ]:
batch_size = 2

blue_chip_dataset = Dataset10x(blue_chip_df, items_path)
tech_dataset = Dataset10x(tech_df, items_path)

# blue_chip_dataloader = DataLoader(blue_chip_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_item7)
# tech_dataloader = DataLoader(tech_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_item7)

In [ ]:
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer=tokenizer,
                                                                     chunk_size=length,
                                                                     chunk_overlap=240)

def split_collator(batch):
    all_chunks = []
    doc_indices = []  # Индекс документа для каждого чанка
    chunks_per_doc = []  # Количество чанков для каждого документа
    
    for idx, data in batch:
        if data is not None and "item7" in data:
            text = data['item7']
            chunks = splitter.split_text(text)
            all_chunks.extend(chunks)
            # Добавляем индекс документа для каждого чанка
            doc_indices.extend([idx] * len(chunks))
            chunks_per_doc.append(len(chunks))
    
    return doc_indices, all_chunks, chunks_per_doc

In [ ]:
blue_chip_dataloader = DataLoader(
    blue_chip_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=split_collator
)

tech_dataloader = DataLoader(
    tech_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=split_collator
)

## Example

In [ ]:
text = "The capital of France is [MASK]."

inputs = tokenizer([text, text+" Another sentence of stuff."], return_tensors="pt", truncation=True, padding='max_length').to(device)
with torch.no_grad():
    outputs = model(**inputs)

masked_index = inputs["input_ids"][0].tolist().index(tokenizer.mask_token_id)
predicted_token_id = outputs.logits[0, masked_index].argmax(axis=-1)
predicted_token = tokenizer.decode(predicted_token_id)
print("Predicted token:", predicted_token)

In [ ]:
item_text = "Some text."

ending = "The capital of France is [MASK]."

prompt = get_prompt(item_text, ending)
probs = get_model_output(prompt, model, device, k=20)
probs

## Report probabilities

In [ ]:
idx = 29

item_dict = blue_chip_dataset[idx]
date = blue_chip_df.iloc[idx].FILING_DATE
item_text = item_dict[1].get(item_name)[:50_000]

co_name = blue_chip_df.iloc[idx]['CoName']

print(f"{co_name}, date: {date}")

In [ ]:
# len(item_dict[1].get(item_name))

In [ ]:
pprint(item_text)

In [ ]:
ending = "In summary, we are [MASK] regarding the future growth of our company."

prompt = get_prompt(item_text, ending)
probs = get_model_output(prompt, model, device, k=20)
probs

In [ ]:
idx = 33

item_dict = blue_chip_dataset[idx]
item_text = item_dict[1].get(item_name)[:50_000]

prompt = get_prompt(item_text, ending)
probs = get_model_output(prompt, model, device)
probs

In [ ]:
idx = 34

item_dict = blue_chip_dataset[idx]
item_text = item_dict[1].get(item_name)[:40_000]

prompt = get_prompt(item_text, ending)
probs = get_model_output(prompt, model, device)
probs

In [ ]:
pprint(item_text)

In [ ]:
clean_mem()

In [ ]:
idx = 1

item_dict = blue_chip_dataset[idx]
item_text = item_dict[1].get(item_name)

prompt = get_prompt(item_text, ending)
probs = get_model_output(prompt, model, device, k=20)
probs

In [ ]:
clean_mem()

## Prompt strategy

In [ ]:
@dataclass(frozen=True)
class Prompt_Strategy:
    name: str
    verbalizer: dict
    prompt: Callable
    top_p: float = 0.99
        

sentiment_verb = {
    "positive": set(['optimistic', 'confident', 'positive', 'encouraged', 'excited',
                     'enthusiastic', 'hopeful', 'comfortable', 'satisfied', 'pleased', 
                     'relaxed', 'encouraged', 'ambitious', 'pleased', 'favorable', 
                     'assured', 'strong', 
             
                     ]),
    
    "negative": set(['cautious', 'concerned', 'aggressive', 'pessimistic', 'negative',
                     'uncomfortable', 'concerned', 'cautious', 'uncertain', 'unsure',
                     'skeptical', 'worried', 'discouraged', 'anxious', 'frustrated',
                     'confused', 'doubtful', 'unsatisfied', 'disappointed'
                    ])
}

len(sentiment_verb['positive']), len(sentiment_verb['negative'])

In [ ]:
ending = "In summary, we are [MASK] regarding the future growth of our company."

tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, sentiment_verb, text_length=q99)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)

In [ ]:
# ok
ending = "In one word, we are [MASK] regarding the future growth of our company."

tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, sentiment_verb, text_length=q99)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)

In [ ]:
ending = "Regarding future growth, we are [MASK]."

sentiment_verb = {
    "positive": set(['optimistic', 'confident', 'positive', 'encouraged', 'excited',
                     'enthusiastic', 'hopeful', 'comfortable', 'satisfied', 'pleased', 
                     'relaxed', 'encouraged', 'ambitious', 'pleased', 'favorable', 
                     'assured', 'strong', 
             
                     ]),
    
    "negative": set(['cautious', 'concerned', 'aggressive', 'pessimistic', 'negative',
                     'uncomfortable', 'concerned', 'cautious', 'uncertain', 'unsure',
                     'skeptical', 'worried', 'discouraged', 'anxious', 'frustrated',
                     'confused', 'doubtful', 'unsatisfied', 'disappointed'
                    ])
}


tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, sentiment_verb, text_length=q99)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)

In [ ]:
ending = "In one word, the company's next period profit is going to be [MASK]."

sentiment_verb = {
    "positive": set(['good', 'best', 'excellent', 'outstanding', 'exceptional',
                     'healthy', 'strong', 'awesome', 'great', 'fantastic', 'stable',
                     'perfect', 'solid', 'profitable', 'impressive', 'reliable',
                     'thriving', 'optimistic', 'decent', 'positive', 'sustainable',]),
    
    "negative": set(['bad', 'poor', 'terrible', 'risky', 'weak', 'dependent',
                     'unstable', 'unhealthy', 'questionable', 'suffering', 'stressed',
                     'fragile', 'negative', 'unsustainable', 'awful', 'vulnerable',
                     'mediocre', 'horrible', 'precarious', 'declining', 'worsening'])
}

tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, sentiment_verb, text_length=q99)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)

In [ ]:
ending = "We project future growth, though with [MASK] confidence."

confidence_verbalizer = {
    "positive": {
        "high", "strong", "great", "full", "complete", "absolute",
        "total", "utmost", "maximum", "supreme", "unwavering",
        "unshakeable", "firm", "solid", "robust", "considerable"
    },
    
    "negative": {
        "low", "limited", "little", "some", "moderate", "cautious",
        "guarded", "measured", "tempered", "qualified", "conditional",
        "uncertain", "tentative", "hesitant", "wavering", "fragile",
        "minimal", "slight", "negligible", "weak", "diminished"
    }
}

tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, confidence_verbalizer, text_length=q99)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)

In [ ]:
tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, confidence_verbalizer, text_length=q99)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)